# 03-phase1-alpha-distribution

Phase 1 (depth-only growing Transformer) 의 **학습된 α 분포 검토** — Growing 효과 부재의 원인 정량 분석.

- **배경**: PR #34 (Trigger 튜닝) 의 발견 — 4 layer vs 8 layer 의 final loss 가 +0.0004 차이 (사실상 동일). 깊이가 2배 늘어도 학습 효과 X.
- **검증할 가설**:
  1. **초기 4 block α 는 1.0 근처 유지** (초기값 1.0 으로 시작, 학습이 잘 진행됨)
  2. **추가 4 block (block 4-7) α 는 0 근처** (초기값 0.0 으로 시작, 학습으로 거의 안 벗어남 → Growing 효과 부재의 원인 입증)
  3. **시드 invariance**: 시드 무관하게 α 패턴 일관
  4. **block 위치별 magnitude**: 나중 추가될수록 (block 7 > block 4) α magnitude 가 더 작은가?
- **데이터**: TinyShakespeare
- **시드**: [42, 123, 456]
- **작성일**: 2026-05-25
- **연관**: Issue [#37](https://github.com/EinSofINTEREST/GraphLM/issues/37) / 동기 PR [#34](https://github.com/EinSofINTEREST/GraphLM/pull/34)

## 1. 환경 / 시드 / device

In [ ]:
import logging
import sys

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.models.backbone import BackboneConfig, GrowingDecoder
from graphlm.training.loop import TrainConfig, train
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")

## 2. 실험 설정

Base / 재현성 / Trigger 튜닝 노트북과 **동일 hyperparameters** 사용 (단, ε=0.04 — #3 의 권장값). 시드만 변동.

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-phase1-alpha"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]

BATCH_SIZE = 16
BLOCK_SIZE = 128

BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_init_layers=4)
TRAIN_BASE = dict(
    lr=3e-4,
    max_steps=1500,
    max_layers=8,
    trigger_window=100,
    trigger_epsilon=0.04,  # #3 의 권장값 (MIXED, 진짜 plateau 감지에 가까움)
    trigger_cooldown=150,
    trigger_min_history=100,
    device=DEVICE,
)

N_INIT = BACKBONE["n_init_layers"]
print("SEEDS    =", SEEDS)
print("N_INIT   =", N_INIT, "(초기 block 수)")
for k, v in {**BACKBONE, **TRAIN_BASE}.items():
    print(f"  {k:22s} = {v}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 시드별 학습 + final α 수집

각 시드마다 model fresh init → train 1500 step → 학습 종료 후 각 block 의 final α 추출.

In [ ]:
cfg = BackboneConfig(vocab_size=tokenizer.vocab_size, max_seq_len=BLOCK_SIZE, **BACKBONE)
train_cfg = TrainConfig(**TRAIN_BASE)

# {seed: {'result': TrainResult, 'alphas': list[float], 'n_layers': int}}
runs = {}
for seed in SEEDS:
    print(f"--- seed = {seed} ---")
    set_seed(seed)
    model = GrowingDecoder(cfg).to(DEVICE)
    data_iter = iter_random_batches(
        dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=seed
    )
    result = train(model, data_iter, train_cfg)
    alphas = [b.alpha.item() for b in model.blocks]
    runs[seed] = {"result": result, "alphas": alphas, "n_layers": model.n_layers}
    print(
        f"  done: n_layers={model.n_layers}, grow_events={len(result.grow_events)}, "
        f"final_loss≈{result.losses[-1]:.4f}"
    )

## 5. 시드별 / block 위치별 α 표

각 행 = 시드, 각 열 = block 위치 (0 ~ n_layers-1). 초기 block (`< n_init`) 은 1.0 에서 시작, 추가 block (`>= n_init`) 은 0.0 에서 시작.

In [ ]:
# 최대 n_layers 결정
max_L = max(r["n_layers"] for r in runs.values())

# header
header_cells = [f"b{i}" + ("*" if i < N_INIT else "+") for i in range(max_L)]
print(f"{'seed':>5}  " + "  ".join(f"{h:>7}" for h in header_cells))
print("-" * (7 + 9 * max_L))
for seed, r in runs.items():
    cells = []
    for i in range(max_L):
        cells.append(f"{r['alphas'][i]:>+7.4f}" if i < r["n_layers"] else f"{'—':>7}")
    print(f"{seed:>5}  " + "  ".join(cells))
print()
print(f"legend: b{0}*-b{N_INIT - 1}* = 초기 block (α=1.0 에서 시작),")
print(f"        b{N_INIT}+-b{max_L - 1}+ = 추가 block (α=0.0 에서 시작, 학습으로 변동)")

## 6. 초기 block vs 추가 block α magnitude 분포

**핵심 분석**: 추가 block 의 α magnitude (|α|) 가 작으면 → Growing 효과 부재의 원인 입증.

In [ ]:
import statistics

# 시드 전체에서 초기 block α / 추가 block α 수집
init_alphas = []  # 초기 block 의 |α|
added_alphas = []  # 추가 block 의 |α|
for r in runs.values():
    for i, a in enumerate(r["alphas"]):
        (init_alphas if i < N_INIT else added_alphas).append(abs(a))


def _stats(values):
    if not values:
        return "(none)"
    return (
        f"n={len(values):>2}  mean={statistics.mean(values):.4f}  "
        f"min={min(values):.4f}  max={max(values):.4f}  "
        f"σ={statistics.stdev(values) if len(values) > 1 else 0:.4f}"
    )


print("초기 block (α=1.0 에서 시작) — |α| 통계:")
print(f"  {_stats(init_alphas)}")
print()
print("추가 block (α=0.0 에서 시작) — |α| 통계:")
print(f"  {_stats(added_alphas)}")
print()

# 비율 분석 — 추가 block 이 초기 block 의 몇 % magnitude 인지
if init_alphas and added_alphas:
    init_mean = statistics.mean(init_alphas)
    if init_mean == 0:
        # 초기 block α 가 모두 0 — 극히 드물지만 ZeroDivisionError 차단
        print("비율: N/A (초기 block 의 mean |α| 가 0 — 학습 비정상 의심)")
    else:
        ratio = statistics.mean(added_alphas) / init_mean
        print(f"비율 (mean |α_added| / mean |α_init|) : {ratio:.3f}")
        print(f"  → 추가 block 의 평균 |α| 가 초기 block 의 {ratio:.1%} 수준")
        if ratio < 0.1:
            verdict = "GROWING 효과 부재 가설 ✅ 입증 (추가 block α 가 거의 0)"
        elif ratio < 0.3:
            verdict = "GROWING 효과 미미 — 추가 block 의 기여도가 초기 block 대비 매우 낮음"
        elif ratio < 0.7:
            verdict = (
                "GROWING 부분 활용 — 추가 block 이 의미 있게 학습되지만 초기 block 만큼 강하지 않음"
            )
        else:
            verdict = "GROWING 효과 정상 — 추가 block α 가 초기 block 수준으로 학습됨"
        print(f"  verdict: {verdict}")

## 7. 시각화 — block 위치별 |α| bar plot (시드별)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
width = 0.25
for i, (seed, r) in enumerate(runs.items()):
    xs = [j + (i - 1) * width for j in range(r["n_layers"])]
    ys = [abs(a) for a in r["alphas"]]
    ax.bar(xs, ys, width=width, label=f"seed={seed}", alpha=0.85)
ax.axvline(N_INIT - 0.5, color="gray", linestyle="--", lw=1, alpha=0.6)
ax.text(N_INIT - 0.5, ax.get_ylim()[1] * 0.95, " added blocks ->", color="gray", fontsize=9)
ax.set_xlabel("block index")
ax.set_ylabel("|alpha| (final)")
ax.set_title("Phase 1 final |alpha| per block (initial 0..3, added 4..7)")
ax.set_xticks(range(max_L))
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(OUT_DIR / "alpha_per_block.png", dpi=120)
plt.show()

## 8. block 위치 (추가 순서) vs |α| 추세

**가설 4**: 나중에 추가된 block (block 7) 일수록 학습 step 수가 적어 |α| 가 더 작은가? 
(추가 시점: ε=0.04 의 grow events 분석에서 step 156, 306, 458, 610 정도 — block 7 은 학습 후반에 추가)

In [ ]:
# 시드 평균 — block 위치별 |α|
by_position = {}  # {block_idx: [|α| from each seed]}
for r in runs.values():
    for i, a in enumerate(r["alphas"]):
        by_position.setdefault(i, []).append(abs(a))

print(f"{'block':>5}  {'is_added':>10}  {'mean |α|':>9}  {'σ':>7}  {'n_seeds':>8}")
print("-" * 50)
for i in sorted(by_position):
    vals = by_position[i]
    is_added = "added" if i >= N_INIT else "init"
    mean = statistics.mean(vals)
    sigma = statistics.stdev(vals) if len(vals) > 1 else 0
    print(f"{i:>5}  {is_added:>10}  {mean:>9.4f}  {sigma:>7.4f}  {len(vals):>8}")

## 9. 시드별 final α + grow_events 요약 표

In [ ]:
print(f"{'seed':>5}  {'n_layers':>9}  {'#grow':>6}  {'final_loss':>11}  grow_steps")
print("-" * 70)
for seed, r in runs.items():
    n = min(100, len(r["result"].losses))
    final = sum(r["result"].losses[-n:]) / n if n > 0 else 0.0
    steps = [s for s, _ in r["result"].grow_events]
    print(f"{seed:>5}  {r['n_layers']:>9}  {len(steps):>6}  {final:>11.4f}  {steps}")

## 결과 요약 / Phase 1 보완 시리즈 종합

이 노트북에서 확인할 것:
- §5 의 표: 시드별 block 위치별 α 패턴
- §6 의 verdict: GROWING 효과 부재 가설 검증 결과
- §7 의 bar plot: 시각적 비교
- §8 의 위치별 |α| 추세: 나중 추가될수록 |α| 감소하는가?

**판정 기준** (§6 의 비율 mean|α_added| / mean|α_init|):
- < 0.1 → 가설 ✅ 강력 입증 (추가 block 거의 0)
- 0.1-0.3 → 미미한 활용
- 0.3-0.7 → 부분 활용
- ≥ 0.7 → 정상 — 깊은 모델이 효과 못 본 다른 원인 (학습 부족 / dataset 한계)

**Phase 1 보완 시리즈 종료** (#1 GPU / #2 재현성 / #3 Trigger 튜닝 / #4 α 분포). 본 노트북의 결과에 따라:
- 가설 입증 → Phase 1 의 본질적 한계 확정 → **Phase 2 (LiGO width growth)** 가 필요한 진짜 근거
- 가설 반박 → 다른 원인 (학습 부족 등) 추가 분석 필요